# Model training and evaluation


In [1]:
%pip install streamlit
%pip install -q scikit-learn sentence-transformers tqdm
%pip install pandas numpy


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Import necessary dependencies


In [2]:
import json, os
from collections import Counter

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sentence_transformers import SentenceTransformer


/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_group/assessment_1/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1127 00:21:36.528000 93808 .venv/lib/python3.15/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Define Constants


In [3]:
CONFIG = json.load(open("config.json"))
output_dir = CONFIG["output_dir"]

MIN_TAG_FREQUENCY = 5
TOP_K             = [5, 10]
MAX_EVAL_QUERIES  = 500
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"

os.makedirs(output_dir, exist_ok=True)


## Load the clean dataset


In [4]:
questions_df = pd.read_csv(CONFIG["clean_path"])
print(f"loaded {len(questions_df)} rows from {CONFIG['clean_path']}")
questions_df["question_id"] = questions_df["question_id"].astype(str)
questions_df["creation_date"] = pd.to_datetime(questions_df["creation_date"], utc=True)
for column_name in ["is_answered", "has_accepted_answer"]:
    questions_df[column_name] = questions_df[column_name].astype(str).str.lower().map({"true": True, "false": False})
for column_name in ["score", "view_count", "answer_count"]:
    questions_df[column_name] = pd.to_numeric(questions_df[column_name], errors="coerce").fillna(0).astype(int)


loaded 16137 rows from dataset/questions_clean.csv


## Prepare labels from the domain tags


In [5]:
# AI/ML tag vocabulary and one multi-hot label row per question
domain_tag_set = set(CONFIG["domain_tags"])
questions_df["domain_tag_list"] = questions_df["tags"].fillna("").str.split("|").map(
    lambda tag_list: sorted(set(tag_list) & domain_tag_set))
tag_counts = Counter(tag for tag_list in questions_df["domain_tag_list"] for tag in tag_list)
tag_vocabulary = sorted(tag for tag, count in tag_counts.items() if count >= MIN_TAG_FREQUENCY)
tag_to_index = {tag: index for index, tag in enumerate(tag_vocabulary)}
print(f"tag vocabulary: {len(tag_vocabulary)} tags with freq >= {MIN_TAG_FREQUENCY}")
print("top tags: " + str(tag_counts.most_common(10)))

label_matrix = np.zeros((len(questions_df), len(tag_vocabulary)), dtype=np.int8)
for row_index, tag_list in enumerate(questions_df["domain_tag_list"]):
    for tag in tag_list:
        if tag in tag_to_index:
            label_matrix[row_index, tag_to_index[tag]] = 1
print(f"label matrix: {label_matrix.shape}, avg tags/question: {label_matrix.sum(axis=1).mean():.2f}")


tag vocabulary: 178 tags with freq >= 5
top tags: [('numpy', 2263), ('matplotlib', 1751), ('pytorch', 1510), ('machine-learning', 1237), ('tensorflow', 1082), ('opencv', 727), ('langchain', 708), ('jupyter-notebook', 707), ('scipy', 673), ('large-language-model', 673)]
label matrix: (16137, 178), avg tags/question: 1.55


## Split the dataset into train, validation and test sets


In [6]:
RANDOM_SPLIT_SEED = 24
train_df, remaining_df = train_test_split(questions_df, test_size=0.2, random_state=RANDOM_SPLIT_SEED)
val_df, test_df = train_test_split(remaining_df, test_size=0.5, random_state=RANDOM_SPLIT_SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
assert not (set(train_df["question_id"]) & set(test_df["question_id"]))

num_eval_queries = min(MAX_EVAL_QUERIES, len(test_df))


split: train=12909, val=1614, test=1614


## Train the models


### Tune the TF-IDF hyperparameters


In [7]:
def make_vectorizer(word_ngrams, char_ngrams, min_df, max_features, stop_words):
    word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=word_ngrams, min_df=min_df,
                                      max_features=max_features, sublinear_tf=True, stop_words=stop_words)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=char_ngrams, min_df=min_df,
                                      max_features=max_features, sublinear_tf=True)
    return FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])

def validation_precision_at_5(vectorizer, k=5):
    train_vectors = vectorizer.fit_transform(train_df["text_all"])
    query_vectors = vectorizer.transform(val_df["text_all"])
    similarity_matrix = cosine_similarity(query_vectors, train_vectors)
    hits = 0
    for query_index in range(len(val_df)):
        top_indices = np.argsort(-similarity_matrix[query_index])[:k]
        query_tags = set(val_df["domain_tag_list"].iloc[query_index])
        hits += sum(1 for catalog_index in top_indices
                    if set(train_df["domain_tag_list"].iloc[catalog_index]) & query_tags)
    return hits / (k * len(val_df))

tfidf_param_grid = [
    {"word_ngrams": (1, 1), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 3, "max_features": 50000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": "english"},
]
best_val_precision, best_tfidf_params = -1.0, None
for params in tfidf_param_grid:
    val_precision = validation_precision_at_5(make_vectorizer(**params))
    print(f"grid {params} -> val precision@5 = {val_precision:.3f}")
    if val_precision > best_val_precision:
        best_val_precision, best_tfidf_params = val_precision, params
print(f"best TF-IDF params (val precision@5={best_val_precision:.3f}): {best_tfidf_params}")


grid {'word_ngrams': (1, 1), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.609
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.606
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 3, 'max_features': 50000, 'stop_words': None} -> val precision@5 = 0.604
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': 'english'} -> val precision@5 = 0.610
best TF-IDF params (val precision@5=0.610): {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': 'english'}


### Vectorize the questions with TF-IDF


In [8]:
vectorizer = make_vectorizer(**best_tfidf_params)
tfidf_train_matrix = vectorizer.fit_transform(train_df["text_all"])
tfidf_val_matrix = vectorizer.transform(val_df["text_all"])
tfidf_test_matrix = vectorizer.transform(test_df["text_all"])
print(f"TF-IDF final (best params): train {tfidf_train_matrix.shape}, "
      f"val {tfidf_val_matrix.shape}, test {tfidf_test_matrix.shape}")


TF-IDF final (best params): train (12909, 195260), val (1614, 195260), test (1614, 195260)


### Generate the sentence embeddings


In [9]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# embeddings: train catalog, test queries, val queries, full catalog (used by the app)
train_embeddings = embedding_model.encode(train_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
test_embeddings = embedding_model.encode(test_df.iloc[:num_eval_queries]["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
val_embeddings = embedding_model.encode(val_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
all_embeddings = embedding_model.encode(questions_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)


Batches: 100%|██████████| 505/505 [00:19<00:00, 26.20it/s]


## Evaluate the models


### Evaluate TF-IDF on the test set


In [10]:
def evaluate_similarity(similarity_matrix, query_rows, catalog_rows, k):
    """precision@k / hit-rate@k / MRR / primary-tag precision@k + random baseline,
    for a similarity matrix (n_queries x n_catalog)."""
    query_tag_lists = query_rows["domain_tag_list"].tolist()
    catalog_tag_lists = catalog_rows["domain_tag_list"].tolist()
    query_primary_tags = [tags.split("|")[0] for tags in query_rows["tags"]]
    catalog_primary_tags = [tags.split("|")[0] for tags in catalog_rows["tags"]]
    precision_scores, hit_rates, reciprocal_ranks, primary_precision_scores, random_baselines = [], [], [], [], []
    for query_index in range(len(query_rows)):
        top_indices = np.argsort(-similarity_matrix[query_index])[:k]
        query_tags = set(query_tag_lists[query_index])
        relevant_flags = [bool(set(catalog_tag_lists[catalog_index]) & query_tags) for catalog_index in top_indices]
        precision_scores.append(np.mean(relevant_flags))
        hit_rates.append(any(relevant_flags))
        for rank, is_relevant in enumerate(relevant_flags, start=1):
            if is_relevant:
                reciprocal_ranks.append(1 / rank)
                break
        else:
            reciprocal_ranks.append(0.0)
        primary_precision_scores.append(
            np.mean([catalog_primary_tags[catalog_index] == query_primary_tags[query_index]
                     for catalog_index in top_indices]))
        num_relevant_in_catalog = sum(1 for catalog_tags in catalog_tag_lists if set(catalog_tags) & query_tags)
        random_baselines.append(num_relevant_in_catalog / len(catalog_tag_lists))
    return (float(np.mean(precision_scores)), float(np.mean(hit_rates)), float(np.mean(reciprocal_ranks)),
            float(np.mean(primary_precision_scores)), float(np.mean(random_baselines)))

MODEL_RESULTS = {}

# TF-IDF retrieval: test queries against the train catalog
tfidf_similarity_matrix = cosine_similarity(tfidf_test_matrix[:num_eval_queries], tfidf_train_matrix)

for k in TOP_K:
    precision, hit_rate, mrr, primary_precision, random_baseline = evaluate_similarity(
        tfidf_similarity_matrix, test_df.iloc[:num_eval_queries], train_df, k)
    MODEL_RESULTS.setdefault("tfidf", {})[k] = (precision, hit_rate, mrr, primary_precision, random_baseline)
    print(f"TF-IDF      @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
          f"primary={primary_precision:.3f} (random={random_baseline:.3f})")

# qualitative sanity check: top-5 for the first 3 test questions
train_tag_lists = train_df["domain_tag_list"].tolist()
for query_index in range(min(3, num_eval_queries)):
    top_indices = np.argsort(-tfidf_similarity_matrix[query_index])[:5]
    print(f"QUERY: {test_df['title'].iloc[query_index][:90]}")
    for catalog_index in top_indices:
        print(f"  {tfidf_similarity_matrix[query_index, catalog_index]:.3f}  "
              f"{train_df['title'].iloc[catalog_index][:80]}  [{', '.join(train_tag_lists[catalog_index])}]")


TF-IDF      @5: precision=0.620 hit=0.890 MRR=0.755 primary=0.447 (random=0.073)
TF-IDF      @10: precision=0.600 hit=0.940 MRR=0.761 primary=0.442 (random=0.073)
QUERY: Do I need .repeat() or steps_per_epoch when training on a finite TFRecord dataset in Keras
  0.265  Your input ran out of data. I already tried several times but failed, Really app  [tensorflow]
  0.263  Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence  [lstm, tf.keras]
  0.203  Tensorflow Keras model with interleave is having issues fitting  [keras, machine-learning, tensorflow]
  0.192  How dataset.repeat() works with model.fit()?  [keras, tensorflow]
  0.175  Tensorflow dataset splitted sizing parameter problem: Local rendezvous is aborti  [tensorflow, tensorflow-datasets]
QUERY: Modeling Sigmoid Curve with Time-Dependent Steepness in Python
  0.217  I am trying to fit a logistic sigmoid function to data. The data in the y-axis i  [logistic-regression]
  0.213  Logistic curve produced by curve

### Evaluate the sentence embeddings


In [11]:
embedding_similarity_matrix = cosine_similarity(test_embeddings, train_embeddings)

for k in TOP_K:
    precision, hit_rate, mrr, primary_precision, random_baseline = evaluate_similarity(
        embedding_similarity_matrix, test_df.iloc[:num_eval_queries], train_df, k)
    MODEL_RESULTS.setdefault("embeddings", {})[k] = (precision, hit_rate, mrr, primary_precision, random_baseline)
    print(f"embeddings  @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
          f"primary={primary_precision:.3f} (random={random_baseline:.3f})")


embeddings  @5: precision=0.673 hit=0.906 MRR=0.792 primary=0.473 (random=0.073)
embeddings  @10: precision=0.653 hit=0.940 MRR=0.796 primary=0.464 (random=0.073)


### Compare the models and choose the best one


In [12]:
print("=== model comparison (test split) ===")
for model_name, model_scores in MODEL_RESULTS.items():
    for k in sorted(model_scores):
        precision, hit_rate, mrr, primary_precision, random_baseline = model_scores[k]
        print(f"{model_name:10s} @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
              f"primary={primary_precision:.3f} (random={random_baseline:.3f})")
best_model_name = max(MODEL_RESULTS, key=lambda model_name: MODEL_RESULTS[model_name][TOP_K[0]][0])
print(f"best model by precision@{TOP_K[0]}: {best_model_name}")


=== model comparison (test split) ===
tfidf      @5: precision=0.620 hit=0.890 MRR=0.755 primary=0.447 (random=0.073)
tfidf      @10: precision=0.600 hit=0.940 MRR=0.761 primary=0.442 (random=0.073)
embeddings @5: precision=0.673 hit=0.906 MRR=0.792 primary=0.473 (random=0.073)
embeddings @10: precision=0.653 hit=0.940 MRR=0.796 primary=0.464 (random=0.073)
best model by precision@5: embeddings


## Save the model artifacts for the app


In [13]:
output_dir = CONFIG["output_dir"]
save_npz(f"{output_dir}/X_train.npz", tfidf_train_matrix)
save_npz(f"{output_dir}/X_val.npz", tfidf_val_matrix)
save_npz(f"{output_dir}/X_test.npz", tfidf_test_matrix)
np.savez_compressed(f"{output_dir}/Y.npz", Y=label_matrix)
json.dump({"tag2idx": tag_to_index}, open(f"{output_dir}/tag_vocab.json", "w"), indent=1)
joblib.dump(vectorizer, f"{output_dir}/vectorizer.joblib")
np.savez_compressed(f"{output_dir}/embeddings_all.npz", E=all_embeddings)
json.dump({"emb_model": EMBEDDING_MODEL,
           "tfidf_params": {key: (list(value) if isinstance(value, tuple) else value)
                            for key, value in best_tfidf_params.items()},
           "model_results": {model_name: {str(k): list(scores) for k, scores in model_scores.items()}
                             for model_name, model_scores in MODEL_RESULTS.items()}},
          open(f"{output_dir}/model_selection.json", "w"), indent=1)
print(f"saved model artifacts to {output_dir}/")


saved model artifacts to outputs/
